## Train PNN models on GLORIA for OLCI/MSI or custom configuration

This notebook trains probablistic neural network (PNN) models to estimate chlorophyll-a concentration (Chla) simulating specific satellite sensors (MSI, OLCI or your own bands).

### What this notebook does:

1. **Loads GLORIA**: Reads hyperspectral remote-ensing reflectance (Rrs) measurements and corresponding Chla values
2. **Simulates satellite sensors**: Selects wavelength bands matching MSI or OLCI sensors by finding the closest GLORIA wavelengths
3. **Prepares datasets**: Merges spectral and Chla datasets, removes samples with missing values in selected bands, splits into train/test sets
4. **Trains PNN models**: Trains one or more probabilistic neural network types (BNN-MCD, BNN-DC, MDN, RNN, or ENS-NN) with:
   - Automatic data scaling (RobustScaler for inputs, MinMaxScaler for log-transformed outputs)
   - Early stopping with patience=80 epochs
   - Multiple replicates for ensemble predictions
5. **Evaluates and saves**: Generates predictions with uncertainty estimates and calculates performance metrics

### Workflow:
Config → Load GLORIA data → Select sensor bands → Clean & split data → Train models → Evaluate & save results

### Requirements:
- The local `pnn` package must be importable (set `project_root` to this repo's path)

In [ ]:
# Change this path to yours
project_root=r'c:\github_repos\eawag\iop_nns'

from pathlib import Path
import sys, numpy as np, pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

import pnn
from pnn import c
from pnn import nn as pnn_nn
from pnn.modeloutput import save_model_outputs
rng = np.random.RandomState(123)

# Verify we're using the correct pnn
print(f"Using pnn from: {pnn.__file__}")
# ignore warnings




Using pnn from: c:\github_repos\eawag\iop_nns\pnn\__init__.py


# Parameter config - adapt to your needs

In [5]:
sensor='MSI' # any of 'MSI', 'OLCI', 'custom'
# Custom sensor bands (only used if sensor='custom')
custom_bands = {
    'B1': 443,
    'B2': 490,
    'B3': 560,
    'B4': 665,
    'B5': 705,
    # Add as many bands as GLORIA supports - check in-situ file below
}
srf_path='' # path to the sensor spectral response functions (SRFs)
recal_split=0.0
targets=['Chla']
bands=[]
nns=['bnn_dc', 'bnn_mcd']
replicates=1 # the number of independent models making predictions, in the paper 25
ensemble_size=10 # this is the number of models if nns ['ens_nn']
val_split=0.1
epochs=1000
batch=64
models_out=r'C:\github_repos\eawag\iop_nns\datasets_train_test\custom_example_out\models'
estimates_out=r'C:\github_repos\eawag\iop_nns\datasets_train_test\custom_example_out\estimates'
label=None
split_timeout=1  # Maximum time in minutes for WD/OOD optimisation

In [7]:
# Create output directories if they don't exist
output_base = Path(r'C:\github_repos\eawag\iop_nns\datasets_train_test\custom_example_out')
output_base.mkdir(parents=True, exist_ok=True)

models_dir = output_base / 'models'
estimates_dir = output_base / 'estimates'

models_dir.mkdir(parents=True, exist_ok=True)
estimates_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directories created:")
print(f"  Models: {models_dir}")
print(f"  Estimates: {estimates_dir}")

Output directories created:
  Models: C:\github_repos\eawag\iop_nns\datasets_train_test\custom_example_out\models
  Estimates: C:\github_repos\eawag\iop_nns\datasets_train_test\custom_example_out\estimates


In [17]:
def _get_sensor_bands(sensor, custom_bands=None):
    """
    Get target wavelengths for each sensor.
    
    Args:
        sensor: 'MSI', 'OLCI', or 'custom'
        custom_bands: dict of band names and wavelengths (only for sensor='custom')
    """
    
    msi_bands = {
        'B1': 443,   # Coastal aerosol
        'B2': 490,   # Blue
        'B3': 560,   # Green
        'B4': 665,   # Red
        'B5': 705,   # Red edge 1
        'B6': 740,   # Red edge 2
    }
    
    olci_bands = {
        'Oa01': 400,
        'Oa02': 412,
        'Oa03': 443,
        'Oa04': 490,
        'Oa05': 510,
        'Oa06': 560,
        'Oa07': 620,
        'Oa08': 665,
        'Oa09': 674,
        'Oa10': 681,
        'Oa11': 709,
        'Oa12': 754,
        'Oa16': 778,
    }
    
    sensors = {
        'MSI': msi_bands,
        'OLCI': olci_bands,
    }
    
    if sensor == 'custom':
        if custom_bands is None or not custom_bands:
            raise ValueError("For sensor='custom', you must provide custom_bands dict. "
                           "Example: custom_bands = {'B1': 443, 'B2': 560, 'B3': 665}")
        print(f"Using CUSTOM sensor with {len(custom_bands)} bands")
        return custom_bands
    
    if sensor not in sensors:
        raise ValueError(f"Unknown sensor '{sensor}'. Choose from: {list(sensors.keys()) + ['custom']}")
    
    return sensors[sensor]

def _find_closest_gloria_wavelengths(df, target_wavelengths):
    """
    Find the closest GLORIA Rrs_mean_XXX columns to target wavelengths.
    
    Args:
        df: DataFrame with GLORIA data
        target_wavelengths: dict or list of target wavelengths in nm
        
    Returns:
        list of column names that are closest matches
    """
    import re
    
    # Extract available GLORIA wavelengths from column names
    gloria_cols = {}
    for col in df.columns:
        if col.startswith('Rrs_mean_'):
            match = re.search(r'Rrs_mean_(\d+(?:\.\d+)?)', col)
            if match:
                wl = float(match.group(1))
                gloria_cols[wl] = col
    
    if not gloria_cols:
        raise ValueError("No Rrs_mean_XXX columns found in dataframe")
    
    available_wls = np.array(sorted(gloria_cols.keys()))
    
    # Handle both dict and list inputs
    if isinstance(target_wavelengths, dict):
        target_wls = list(target_wavelengths.values())
    else:
        target_wls = target_wavelengths
    
    # Find closest matches
    selected_cols = []
    for target_wl in target_wls:
        idx = np.argmin(np.abs(available_wls - target_wl))
        closest_wl = available_wls[idx]
        selected_cols.append(gloria_cols[closest_wl])
        print(f"Target {target_wl} nm → Using GLORIA {closest_wl} nm ({gloria_cols[closest_wl]})")
    
    return selected_cols

In [12]:
# load the GLORIA dataset and merge the Chla column with the Rrs_mean_XXX columns

train_path = r'C:\github_repos\eawag\iop_nns\datasets_train_test\GLORIA_2022'
gloria_chl = pd.read_csv(train_path + r'\GLORIA_meta_and_lab.csv')
rrs_mean = pd.read_csv(train_path + r'\GLORIA_Rrs_mean.csv')

# Merge the Chla column from gloria_chl with all columns from rrs_mean
rrs_chl_gloria = pd.concat([gloria_chl[['Chla', 'Site_name']], rrs_mean], axis=1).dropna(subset=['Chla', 'Rrs_mean_415']).reset_index(drop=True)

In [15]:
# first 5 rows
rrs_chl_gloria.head(5)

,Chla,Site_name,GLORIA_ID,N,Rrs_mean_350,Rrs_mean_351,Rrs_mean_352,Rrs_mean_353,Rrs_mean_354,Rrs_mean_355,...,Rrs_mean_891,Rrs_mean_892,Rrs_mean_893,Rrs_mean_894,Rrs_mean_895,Rrs_mean_896,Rrs_mean_897,Rrs_mean_898,Rrs_mean_899,Rrs_mean_900
0,1.243,Blowering Reservoir,GID_192,19.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000021,0.000021,0.000021,0.000021,0.000021,0.000020,0.000020,0.000020,0.000020,0.000020
1,8.414,Blowering Reservoir,GID_194,9.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000029,0.000029,0.000030,0.000029,0.000029,0.000028,0.000027,0.000026,0.000025,0.000025
2,4.975,Blowering Reservoir,GID_196,11.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000035,0.000037,0.000040,0.000039,0.000038,0.000037,0.000037,0.000036,0.000035,0.000035
3,3.008,Blowering Reservoir,GID_198,7.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000040,0.000043,0.000047,0.000066,0.000086,0.000108,0.000108,0.000095,0.000082,0.000072
4,3.400,Blowering Reservoir,GID_199,17.0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000025,0.000025,0.000025,0.000025,0.000026,0.000026,0.000027,0.000027,0.000028,0.000028


In [ ]:
# After creating rrs_chl_gloria, prepare datasets and create all three splits

# First, get the sensor bands to determine which columns will be used
if sensor:
    target_wavelengths = _get_sensor_bands(sensor, custom_bands=custom_bands)
    # Get a preliminary list of columns (before filtering for NaN)
    spectral_cols_preliminary = _find_closest_gloria_wavelengths(rrs_chl_gloria, target_wavelengths)
else:
    raise ValueError("sensor parameter must be set to 'MSI', 'OLCI', or 'custom'")

print(f"Original data size: {len(rrs_chl_gloria)}")

# Drop rows with NaN ONLY in the selected spectral bands + Chla
cols_to_check = spectral_cols_preliminary + ['Chla']
rrs_chl_gloria = rrs_chl_gloria.dropna(subset=cols_to_check).reset_index(drop=True)

print(f"After removing NaN values in selected bands: {len(rrs_chl_gloria)}")
print(f"Columns checked for NaN: {cols_to_check}")

# ============================================================================
# CREATE THREE DATA SPLITS: Random, WD, and OOD
# ============================================================================

# Import the existing splitting functions from dataset_split.py
import dataset_split
from dataset_split import random_split, system_wd_split, system_ood_split

print("\n" + "="*70)
print("CREATING THREE DATA SPLITS")
print("="*70)

# Verify that Site_name column exists
if 'Site_name' not in rrs_chl_gloria.columns:
    raise ValueError("'Site_name' column not found in data. "
                    f"Available columns: {rrs_chl_gloria.columns.tolist()}")

# Update the summary_cols in dataset_split to use Chla for scoring
original_summary_cols = dataset_split.summary_cols
dataset_split.summary_cols = ['Chla']  # Use Chla for WD/OOD similarity scoring

# 1. Random split
print("\n1. Creating RANDOM split...")
train_random, test_random = random_split(rrs_chl_gloria, test_fraction=0.5, seed=42)
print(f"   Train: {len(train_random)} samples")
print(f"   Test: {len(test_random)} samples")

# 2. Within-Distribution (WD) split
print("\n2. Creating WITHIN-DISTRIBUTION (WD) split...")
print(f"   Splitting by 'Site_name', optimizing for SIMILAR Chla distributions...")
print(f"   Maximum optimisation time: {split_timeout} minutes")
train_wd, test_wd = system_wd_split(rrs_chl_gloria, 'Site_name', 
                                     test_fraction=0.5, seed=42, timeout=split_timeout)
print(f"   Train: {len(train_wd)} samples from {train_wd['Site_name'].nunique()} sites")
print(f"   Test: {len(test_wd)} samples from {test_wd['Site_name'].nunique()} sites")
print(f"   Train Chla range: [{train_wd['Chla'].min():.2f}, {train_wd['Chla'].max():.2f}]")
print(f"   Test Chla range: [{test_wd['Chla'].min():.2f}, {test_wd['Chla'].max():.2f}]")

# 3. Out-of-Distribution (OOD) split
print("\n3. Creating OUT-OF-DISTRIBUTION (OOD) split...")
print(f"   Splitting by 'Site_name', optimizing for DISSIMILAR Chla distributions...")
print(f"   Maximum optimisation time: {split_timeout} minutes")
train_ood, test_ood = system_ood_split(rrs_chl_gloria, 'Site_name', 
                                        test_fraction=0.5, seed=42, timeout=split_timeout)
print(f"   Train: {len(train_ood)} samples from {train_ood['Site_name'].nunique()} sites")
print(f"   Test: {len(test_ood)} samples from {test_ood['Site_name'].nunique()} sites")
print(f"   Train Chla range: [{train_ood['Chla'].min():.2f}, {train_ood['Chla'].max():.2f}]")
print(f"   Test Chla range: [{test_ood['Chla'].min():.2f}, {test_ood['Chla'].max():.2f}]")

# Restore original summary_cols
dataset_split.summary_cols = original_summary_cols

print("\n" + "="*70)
print("ALL THREE SPLITS CREATED SUCCESSFULLY")
print("="*70)

Target 443 nm → Using GLORIA 443.0 nm (Rrs_mean_443)
Target 490 nm → Using GLORIA 490.0 nm (Rrs_mean_490)
Target 560 nm → Using GLORIA 560.0 nm (Rrs_mean_560)
Target 665 nm → Using GLORIA 665.0 nm (Rrs_mean_665)
Target 705 nm → Using GLORIA 705.0 nm (Rrs_mean_705)
Target 740 nm → Using GLORIA 740.0 nm (Rrs_mean_740)
Original data size: 1566
After removing NaN values in selected bands: 1218
Columns checked for NaN: ['Rrs_mean_443', 'Rrs_mean_490', 'Rrs_mean_560', 'Rrs_mean_665', 'Rrs_mean_705', 'Rrs_mean_740', 'Chla']

CREATING THREE DATA SPLITS

1. Creating RANDOM split...
   Train: 609 samples
   Test: 609 samples

2. Creating WITHIN-DISTRIBUTION (WD) split...
   Splitting by 'Site_name', optimizing for SIMILAR Chla distributions...
   Maximum optimization time: 1 minutes
00:00 - Minimum #  1. Objective function value:  45.0969
00:00 - Minimum #  2. Objective function value:  29.1004
00:00 - Minimum #  3. Objective function value:  14.9432
00:01 - Minimum #  4. Objective function valu

### SRF handling of OLCI/MSI

In [ ]:
# skipped here, needs implementation

In [19]:
def _prepare_xy(df, bands, targets):
    """
    Prepare X and y from a single DataFrame containing both spectral and target data.
    
    Args:
        df: DataFrame with both spectral columns and target columns
        bands: list of band/spectral column names to use as features
        targets: list of target column names (e.g., ['Chla'])
    """
    if not targets:
        raise ValueError('Need at least one target')
    
    # Check that targets exist
    for t in targets:
        if t not in df.columns:
            raise ValueError(f'Target {t} not found in dataframe')
    
    # Check that bands exist
    for b in bands:
        if b not in df.columns:
            raise ValueError(f'Band {b} not found in dataframe')
    
    X = df[bands].to_numpy(dtype=float)
    y = df[targets].to_numpy(dtype=float)
    
    if (y <= 0).any():
        raise ValueError('Targets must be strictly positive')
    
    return X, y

def _fit_scalers(X, y):
    sx = RobustScaler().fit(X)
    sy = MinMaxScaler().fit(np.log(y))
    return sx, sy

def _select_param(name):
    m = {'bnn_dc': c.bnn_dc, 'bnn_mcd': c.bnn_mcd, 'ens_nn': c.ensemble, 'mdn': c.mdn, 'rnn': c.rnn}
    if name not in m:
        raise ValueError(f'Unknown nn {name}')
    return m[name]

def _train_and_evaluate_one(nn_name, Xtr, ytr, Xte, yte, Xre, yre, sx, sy, label, 
                           models_out, estimates_out, val_split, epochs, batch, 
                           replicates, ensemble_size, targets):
    param = _select_param(nn_name)
    PNN = pnn_nn.select_nn(param)
    models = []
    
    if nn_name == 'ens_nn':
        model = PNN.build_and_train(Xtr, ytr, scaler_X=sx, scaler_y=sy, N=ensemble_size)
        models = [model]
    else:
        models = pnn_nn.train_N_models(PNN, Xtr, ytr, n_models=replicates, scaler_X=sx, 
                                      scaler_y=sy, build_kwargs={}, epochs=epochs, 
                                      batch_size=batch, validation_split=val_split, verbose=1)
    
    if Xre is not None and yre is not None:
        models = pnn_nn.recalibrate_pnn(models, Xre, yre)
    
    save_dir = Path(models_out) / nn_name / label
    save_dir.mkdir(parents=True, exist_ok=True)
    pnn_nn.save_models(models, save_dir / f'{nn_name}_{label}.zip')
    
    metrics = []
    for i, m in enumerate(models, 1):
        mu, tot, ale, epi = m.predict_with_uncertainty(Xte)
        save_model_outputs(yte, mu, tot, ale, epi, 
                          filename=(Path(estimates_out) / f'{nn_name}_{label}_model{i}.csv'), 
                          columns=targets)
        df = pnn_nn.calculate_metrics(yte, mu, tot, columns=targets)
        df.insert(0, 'nn', nn_name)
        df.insert(1, 'replicate', i)
        metrics.append(df)
    
    met = pd.concat(metrics, axis=0).reset_index()
    met.to_csv(Path(estimates_out) / f'{nn_name}_{label}_metrics.csv', index=False)
    return met

In [ ]:
# ============================================================================
# TRAIN ON ALL THREE SPLITS: Random, WD, and OOD
# ============================================================================

# Define all splits to train on
all_splits = {
    'random': (train_random, test_random, "Random"),
    'wd': (train_wd, test_wd, "Within-Distribution"),
    'ood': (train_ood, test_ood, "Out-of-Distribution")
}

# Store all metrics from all splits
all_metrics = []

# Loop through each split
for split_key, (train_data, test_data, split_name) in all_splits.items():
    print("\n" + "=" * 70)
    print(f"TRAINING ON {split_name.upper()} SPLIT")
    print("=" * 70)
    print(f"Train: {len(train_data)} samples, Test: {len(test_data)} samples\n")
    
    # Get spectral column names based on sensor choice
    if sensor:
        target_wavelengths = _get_sensor_bands(sensor, custom_bands=custom_bands)
        spectral_cols = _find_closest_gloria_wavelengths(train_data, target_wavelengths)
    else:
        raise ValueError("sensor parameter must be set to 'MSI', 'OLCI', or 'custom'")
    # Prepare X and y for train and test
    Xtr, ytr = _prepare_xy(train_data, spectral_cols, targets)
    Xte, yte = _prepare_xy(test_data, spectral_cols, targets)
    
    # Optional: recalibration set (if needed)
    Xre = yre = None
    
    # Fit scalers
    sx, sy = _fit_scalers(Xtr, ytr)
    
    # Set up output paths with split name
    models_dir = Path(models_out) if models_out else c.model_path
    est_dir = Path(estimates_out) if estimates_out else c.model_estimates_path
    
    # Add split subdirectories
    models_dir_split = models_dir / split_key
    est_dir_split = est_dir / split_key
    
    models_dir_split.mkdir(parents=True, exist_ok=True)
    est_dir_split.mkdir(parents=True, exist_ok=True)
    
    # Create label with split name
    label_use = label if label else f"{sensor}_GLORIA_{split_key}_{'-'.join(targets)}"
    
    print(f"Saving outputs to:")
    print(f"  Models: {models_dir_split}")
    print(f"  Estimates: {est_dir_split}")
    print(f"  Label: {label_use}\n")
    
    # Train and evaluate
    split_metrics = []
    for nn_name in nns:
        met = _train_and_evaluate_one(nn_name, Xtr, ytr, Xte, yte, Xre, yre, sx, sy, 
                                       label_use, models_dir_split, est_dir_split, val_split, 
                                       epochs, batch, replicates, ensemble_size, targets)
        # Add split identifier to metrics
        met['split'] = split_name
        met['split_key'] = split_key
        split_metrics.append(met)
    
    # Save metrics for this split
    split_metrics_df = pd.concat(split_metrics, axis=0)
    split_metrics_df.to_csv(est_dir_split / f"metrics_{label_use}.csv", index=False)
    
    # Add to overall metrics collection
    all_metrics.append(split_metrics_df)
    
    print(f"\n✓ {split_name} split complete!")

# ============================================================================
# SUMMARY: Combine all metrics
# ============================================================================

print("\n" + "=" * 70)
print("ALL TRAINING COMPLETE!")
print("=" * 70)

# Combine all metrics
metrics_all = pd.concat(all_metrics, axis=0).reset_index(drop=True)

# Save combined metrics
combined_metrics_file = Path(estimates_out) / f"metrics_all_splits_{sensor}_GLORIA_{'-'.join(targets)}.csv"
metrics_all.to_csv(combined_metrics_file, index=False)

print(f"\nCombined metrics saved to: {combined_metrics_file}")
print(f"\nOutput structure:")
print(f"  {estimates_out}/")
print(f"    ├── random/")
print(f"    │   ├── {nns[0]}_*_model*.csv")
print(f"    │   └── metrics_*.csv")
print(f"    ├── wd/")
print(f"    │   ├── {nns[0]}_*_model*.csv")
print(f"    │   └── metrics_*.csv")
print(f"    ├── ood/")
print(f"    │   ├── {nns[0]}_*_model*.csv")
print(f"    │   └── metrics_*.csv")
print(f"    └── metrics_all_splits_*.csv")

print(f"\nOutput CSV files contain multi-index format with:")
print("  - 'category' index: y_true, y_pred, total_var, ale_var, epi_var")
print("  - 'instance' index: sample number (0, 1, 2, ...)")
print("\nTo read back:")
print("  df = pd.read_csv('filename.csv', index_col=['category', 'instance'])")
print("  y_true = df.loc['y_true', 'Chla']")
print("  y_pred = df.loc['y_pred', 'Chla']")
print("=" * 70)

# Display summary metrics
print("\n" + "=" * 70)
print("METRICS SUMMARY BY SPLIT")
print("=" * 70)
metrics_all.head(20)